# 第16章 终身学习 (Lifelong Learning / Continual Learning)

> "人类不会每学一个新技能就忘记所有旧技能，但神经网络会——这就是灾难性遗忘。"

## 1. 知识地图：章节结构与概览

```
16.1 灾难性遗忘
    ├── 什么是灾难性遗忘
    ├── 同一网络学多个任务：学新忘旧
    ├── 与多任务学习的对比
    ├── bAbI 任务案例分析
    ├── 为什么不能一个任务一个模型
    └── 为什么需要终身学习（而非多任务学习）
16.2 终身学习评估方法
    ├── 准确率矩阵 R_{i,j}
    ├── 最终平均准确率公式
    └── 反向迁移（Backward Transfer）
16.3 终身学习的主要解法
    ├── 选择性突触可塑性（基于正则的方法）
    │   ├── 核心思想：重要参数不能大改
    │   ├── 损失函数修改：L' = L + λ Σ b_i (θ_i - θ_i^old)²
    │   ├── 重要性系数 b_i 的确定方法
    │   └── 不妥协（intransigence）问题
    └── 梯度回合记忆（GEM，基于梯度的方法）
        ├── 核心思想：梯度方向不能冲突
        └── 需要存储历史梯度信息
```

## 2. 核心概念：灾难性遗忘（Catastrophic Forgetting）

### 2.1 什么是灾难性遗忘？

**直觉类比：** 想象一个学生，先学会了数学考90分。然后开始学物理，学完之后物理考得很好，但回去考数学只考了80分——不是普通的忘记，而是"灾难性"的断崖式下跌。如果再学化学，数学和物理都大幅下跌。

**实际案例：**
1. 训练模型识别手写数字（MNIST）→ 准确率 90%
2. 让模型学习识别有噪声的手写数字
3. 回头测试干净的手写数字 → 准确率降到 80%！

更严重的是，即使把任务1和任务2的数据放在一起同时训练，模型是能同时学好两个任务的（说明模型容量足够），但顺序学就不行。

### 2.2 bAbI 任务案例分析

bAbI 是自然语言QA任务，有20个子任务。例如任务5："Mary把蛋糕给了Fred"，"Fred把蛋糕给了Bill"，问"谁给了蛋糕给Fred？"。

实验发现：
- **同时学习20个任务**：模型能学会大部分任务（准确率很高）
- **依次学习20个任务**：学到任务5时准确率100%，但开始学任务6后，任务5准确率暴跌——完全忘记

### 2.3 为什么不能直接用多任务学习替代？

- **多任务学习**：把所有任务的数据混在一起同时训练（没有遗忘）
- **问题**：如果1000个任务，学第1000个任务时要重新训练前999个——成本不可接受
- **类比**：每上一门新课，都要把所有学过的课重学一遍，这不是人类的学习方式
- 多任务学习是**终身学习的理论上界**

### 2.4 为什么不能每个任务各训练一个模型？

- 1000个任务 = 1000个模型 → 存储爆炸
- 各任务间的共享知识无法被利用
- 模型增长与任务数量线性增长

## 3. 终身学习评估方法

### 3.1 评估矩阵

定义 $R_{i,j}$ = 学完任务1到任务i后，再测试任务j的准确率。

例如学了3个任务形成的矩阵：

            任务1  任务2  任务3
    学完1:   R11     -      -
    学完2:   R21    R22     -
    学完3:   R31    R32    R33

### 3.2 两个核心指标

**指标1：最终平均准确率**

$$\text{Accuracy} = \frac{1}{T} \sum_{i=1}^{T} R_{T,i}$$

- 学完所有T个任务后，每个任务上的准确率的平均值
- 这是最主要的评估指标

**指标2：反向迁移（Backward Transfer）**

$$\text{BWT} = \frac{1}{T-1} \sum_{i=1}^{T-1} (R_{T,i} - R_{i,i})$$

- 负值 = 灾难性遗忘（新任务损害了旧任务）
- 正值 = 正向迁移（新任务反而帮助了旧任务！理想情况）
- 0 = 没有任何迁移效应

## 4. 灾难性遗忘的几何解释

考虑只有两个参数 $\theta_1, \theta_2$ 的模型：

- **任务1的损失函数**在参数空间形成一个"山谷"，谷底（蓝色区域）对应低损失
- **任务2的损失函数**有另一个"山谷"，位置不同
- 从随机起点 $\theta^0$ 开始，用梯度下降找到任务1的谷底 $\theta^b$
- 然后从 $\theta^b$ 开始训练任务2，参数被梯度拉向任务2的谷底 $\theta^*$
- 但 $\theta^*$ 可能不在任务1的谷底范围内！

**关键认知：** 任务1的低损失区域是一个"区域"而非一个"点"。如果训练任务2时参数只在任务1的低损失区域内移动（而不离开它），就不会发生灾难性遗忘。这就是选择性突触可塑性的几何直觉。

## 5. 解法一：选择性突触可塑性（基于正则）

### 5.1 核心思想

不是所有参数都一样重要！对旧任务重要的参数，在学习新任务时不能大改。

### 5.2 修改后的损失函数

$$L'(\theta) = L(\theta) + \lambda \sum_i b_i (\theta_i - \theta_i^{\text{old}})^2$$

其中：
- $L(\theta)$：新任务的标准损失
- $\theta_i^{\text{old}}$：旧任务训练完成时的参数值
- $b_i$：参数 $\theta_i$ 对旧任务的重要性
- $\lambda$：正则化强度超参数

**参数含义：**
- $b_i = 0$：参数 $\theta_i$ 对旧任务不重要，可以自由更新
- $b_i$ 很大：参数 $\theta_i$ 对旧任务非常关键，不能大改
- $b_i \to \infty$：参数被"冻结"——完全不遗忘旧任务，但也学不会新任务（不妥协 intransigence）

### 5.3 如何确定重要性 $b_i$？

直觉：看损失函数对各参数的"敏感度"。

- 轻微改变某个参数 → 损失剧烈变化 → 这个参数很重要！（陡峭方向）
- 轻微改变某个参数 → 损失几乎不变 → 这个参数不重要（平坦方向）

形式化：用 Fisher Information Matrix 或损失函数的二阶导数（曲率）来衡量。各种方法（如EWC、SI、MAS）的区别主要在于如何估算这个重要性。

In [ ]:
# 弹性权重巩固（EWC）的PyTorch实现
import torch
import torch.nn as nn
import torch.nn.functional as F
import copy

class EWC:
    """
    Elastic Weight Consolidation (EWC)
    核心思想：用 Fisher Information Matrix 估计参数重要性
    """
    def __init__(self, model, lambda_ewc=100.0):
        self.model = model
        self.lambda_ewc = lambda_ewc
        self.old_params = {}     # 旧任务的最优参数 θ^old
        self.fisher = {}          # Fisher信息矩阵（重要性 b_i）
    
    def consolidate(self, dataloader):
        """
        在旧任务的数据上计算 Fisher 信息矩阵
        保存当前参数作为 θ^old
        """
        # 保存当前参数
        for name, param in self.model.named_parameters():
            self.old_params[name] = param.data.clone()
            self.fisher[name] = torch.zeros_like(param.data)
        
        # 计算 Fisher 信息矩阵
        self.model.eval()
        for batch_x, batch_y in dataloader:
            self.model.zero_grad()
            output = self.model(batch_x)
            # 对负对数似然采样
            loss = F.nll_loss(F.log_softmax(output, dim=1), batch_y)
            loss.backward()
            # 累积梯度的平方（Fisher的对角近似）
            for name, param in self.model.named_parameters():
                if param.grad is not None:
                    self.fisher[name] += param.grad.data ** 2 / len(dataloader)
    
    def ewc_loss(self):
        """
        计算 EWC 正则化项
        L_ewc = Σ (F_i/2) * (θ_i - θ_i^old)²
        """
        loss = 0.0
        for name, param in self.model.named_parameters():
            if name in self.old_params:
                loss += (self.fisher[name] * (param - self.old_params[name]) ** 2).sum()
        return self.lambda_ewc * loss
    
    def total_loss(self, task_loss):
        """总损失 = 新任务损失 + EWC正则化项"""
        return task_loss + self.ewc_loss()

print("EWC 实现完成")
print("Fisher 信息矩阵 = 参数重要性 b_i")
print("EWC损失 = λ * Σ F_i * (θ_i - θ_i^old)²")

In [ ]:
# 简化的终身学习训练循环
import torch
import torch.nn as nn
import torch.optim as optim

class SimpleLifelongLearner:
    """
    简化的终身学习框架
    支持基于正则的灾难性遗忘缓解
    """
    def __init__(self, model, lr=0.001, importance_weight=10.0):
        self.model = model
        self.optimizer = optim.Adam(model.parameters(), lr=lr)
        self.importance_weight = importance_weight
        self.task_params = []      # 存储每个任务的最优参数
        self.task_importance = []  # 存储每个任务的重要性向量
    
    def estimate_importance(self, dataloader):
        """通过参数梯度的L2范数估计重要性"""
        importance = {}  
        for name, param in self.model.named_parameters():
            importance[name] = torch.zeros_like(param.data)
        
        self.model.eval()
        for batch_x, batch_y in dataloader:
            self.model.zero_grad()
            output = self.model(batch_x)
            loss = F.cross_entropy(output, batch_y)
            loss.backward()
            for name, param in self.model.named_parameters():
                if param.grad is not None:
                    importance[name] += param.grad.abs()
        return importance
    
    def finish_task(self, dataloader):
        """完成一个任务：保存参数和重要性"""
        importance = self.estimate_importance(dataloader)
        saved_params = {}
        for name, param in self.model.named_parameters():
            saved_params[name] = param.data.clone()
        self.task_params.append(saved_params)
        self.task_importance.append(importance)
    
    def compute_regularization_loss(self):
        """计算所有旧任务的正则化损失"""
        reg_loss = 0.0
        for saved_params, importance in zip(self.task_params, self.task_importance):
            for name, param in self.model.named_parameters():
                if name in saved_params:
                    diff = param - saved_params[name]
                    reg_loss += (importance[name] * diff ** 2).sum()
        return self.importance_weight * reg_loss
    
    def train_on_task(self, dataloader, epochs=5):
        """在单个任务上训练"""
        self.model.train()
        for epoch in range(epochs):
            total_loss = 0.0
            for batch_x, batch_y in dataloader:
                self.optimizer.zero_grad()
                output = self.model(batch_x)
                task_loss = F.cross_entropy(output, batch_y)
                reg_loss = self.compute_regularization_loss()
                total = task_loss + reg_loss
                total.backward()
                self.optimizer.step()
                total_loss += task_loss.item()
            print(f"Epoch {epoch+1}/{epochs}, Loss: {total_loss/len(dataloader):.4f}")
    
    def evaluate_all_tasks(self, test_loaders):
        """评估所有任务的准确率"""
        self.model.eval()
        accuracies = []
        with torch.no_grad():
            for loader in test_loaders:
                correct, total = 0, 0
                for batch_x, batch_y in loader:
                    output = self.model(batch_x)
                    pred = output.argmax(dim=1)
                    correct += (pred == batch_y).sum().item()
                    total += batch_y.size(0)
                accuracies.append(correct / total)
        return accuracies

print("终身学习训练框架定义完成")
print("核心：new_loss = task_loss + λ * Σ importance_i * (θ - θ_i_old)²")

## 6. 解法二：梯度回合记忆（GEM）

### 6.1 核心思想

不在参数空间做限制，而是在**梯度方向**上做限制。

更新新任务时：
1. 计算新任务的梯度 $g_{\text{new}}$
2. 计算每个旧任务的梯度 $g_{\text{old}}^{(k)}$（需要存储旧任务数据或梯度）
3. 如果 $g_{\text{new}}$ 与某个 $g_{\text{old}}^{(k)}$ 方向冲突（内积 < 0），将其投影到不冲突的方向
4. 用投影后的梯度更新参数

### 6.2 约束条件

$$g_{\text{new}}^T g_{\text{old}}^{(k)} \geq 0, \quad \forall k$$

新梯度方向不能与任何历史任务的梯度方向相反，否则会伤害旧任务。

### 6.3 优势与局限

- **优势**：直接约束梯度方向，理论上更精确
- **局限**：需要存储旧任务的样本/梯度信息，部分违背了终身学习"不保留原始数据"的初衷

## 7. 终身学习与其他概念的辨析

### 7.1 终身学习 vs 迁移学习

| 维度 | 终身学习 | 迁移学习 |
|------|------|------|
| 关注点 | 新任务**和**旧任务都要做好 | 只关心新任务做得好不好 |
| 方向 | 双向（前向+后向迁移） | 单向（源→目标） |
| 目标 | 持续学习不遗忘 | 利用旧知识帮助新任务 |

### 7.2 终身学习 vs 多任务学习

- **多任务学习**：同时访问所有任务的数据，一起训练
- **终身学习**：顺序访问任务数据，不能回头
- 多任务学习是终身学习的**性能上界**

### 7.3 终身学习 vs 持续学习 vs 增量学习

这三个词在文献中经常混用，指同一件事：模型顺序学习多个任务而不遗忘。

## 8. 当前局限与未来展望

### 8.1 当前研究的局限性

- 目前大多数终身学习论文中的"不同任务"，实际上只是**同一任务的不同域**
- 例如：任务1 = 手写数字识别（干净），任务2 = 手写数字识别（加噪声），任务3 = 手写数字识别（旋转）
- 离真正的"先学语音识别、后学图像分类、再学机器翻译"还有很大距离

### 8.2 为什么研究"同类任务"也有意义

- 即使是如此相似的任务，灾难性遗忘已经很严重
- 如果连同类任务都搞不定，更复杂的场景更无从谈起
- 真实的工业应用场景中也有"同类任务不同分布"的需求（如模型天天接收新用户数据更新）

## 9. 常见误区与易错点

### 误区 1：灾难性遗忘 = 模型容量不够
**纠正：** 当把所有任务数据混在一起同时训练时，同样大小的模型可以学好所有任务。这说明不是容量问题，而是**优化轨迹**的问题——顺序优化时梯度把参数带到了只能做好最新任务的区域。

### 误区 2：终身学习可以用"多存几个模型"解决
**纠正：** 存储成本和任务间知识无法共享是主要问题。此外，训练时可能不知道任务边界在哪里。

### 误区 3：正则化方法不会影响新任务的学习
**纠正：** 如果正则化太强（$b_i$ 太大），新任务完全学不会——这叫做不妥协（intransigence）。需要在"不遗忘"和"学得会"之间找到平衡。

### 误区 4：终身学习的研究已经成熟
**纠正：** 目前的研究仍局限在非常简单的场景（同类任务不同域），距离真正的"终身学习"还有很大距离。这是一个活跃的开放研究领域。

## 10. 与其他章节的联系

| 章节 | 联系 |
|------|------|
| 第3章 深度学习基础 | 梯度下降的优化轨迹直接导致灾难性遗忘 |
| 第8章 GAN | GAN的交替训练也是某种多任务场景（生成器vs辨别器） |
| 第10章 自监督学习 | 预训练模型在下游任务上的微调也可能遗忘预训练知识 |
| 第15章 元学习 | 都是多任务学习场景——元学习重"快速适应新任务"，终身学习重"不忘旧任务" |
| 第19章 ChatGPT | 大模型持续更新（新知识注入）挑战与终身学习相通 |

## 11. 核心公式汇总

### 最终准确率

$$\text{Accuracy} = \frac{1}{T} \sum_{i=1}^{T} R_{T,i}$$

### 反向迁移

$$\text{BWT} = \frac{1}{T-1} \sum_{i=1}^{T-1} (R_{T,i} - R_{i,i})$$

### 选择性突触可塑性（正则化）损失

$$L'(\theta) = L(\theta) + \lambda \sum_i b_i (\theta_i - \theta_i^{\text{old}})^2$$

- $L(\theta)$：新任务的标准损失
- $b_i$：参数 $\theta_i$ 对旧任务的重要性（Fisher信息/曲率等）
- $\lambda$：平衡"不遗忘"和"学得会"的超参数

### GEM 梯度约束

$$g_{\text{new}}^T g_{\text{old}}^{(k)} \geq 0, \quad \forall k$$

新梯度不能与任何旧任务的梯度方向相反。

## 12. 关键总结

1. **灾难性遗忘是深度学习走向AGI的核心障碍之一**：模型顺序学习多个任务时会快速遗忘前面的知识
2. **不是容量问题，是优化问题**：同时学所有任务就没问题，说明模型容量足够
3. **两种主流解法**：选择性突触可塑性（正则化重要参数）和梯度回合记忆（约束梯度方向）
4. **重要性 $b_i$ 是关键**：它决定了哪些参数可以改、哪些必须保护
5. **在"不遗忘"和"学得会"之间需要平衡**：正则太强学不会新任务（不妥协），太弱忘光旧任务
6. **多任务学习是终身学习的性能上界**
7. **当前研究局限于同类任务的不同域**：离真正"不同任务"的终身学习还有距离
8. **评估需要同时看最终准确率和反向迁移**
9. **终身学习、持续学习、增量学习是同义词**
10. **这个领域仍然是活跃的开放研究问题**